In [37]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, f1_score, mean_squared_error, mean_absolute_error, r2_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EvalPrediction
)
from datasets import Dataset
from tqdm import tqdm
import torch
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [42]:
model = AutoModelForSequenceClassification.from_pretrained(
    "DeepPavlov/rubert-base-cased-sentence", num_labels=1
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at DeepPavlov/rubert-base-cased-sentence and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [43]:
df = pd.read_csv('/kaggle/input/news-1/news.csv')
df = df[['Title', 'Score']].dropna()

train_df, val_df = train_test_split(df, test_size=0.3, random_state=42)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained("DeepPavlov/rubert-base-cased-sentence")

In [44]:
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True)).rename_column("Score", "labels")
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True)).rename_column("Score", "labels")

def tokenize(batch):
    return tokenizer(batch["Title"], padding="max_length", truncation=True, max_length=128)

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)

train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
val_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

Map:   0%|          | 0/222344 [00:00<?, ? examples/s]

Map:   0%|          | 0/95291 [00:00<?, ? examples/s]

In [45]:
def compute_metrics(eval_pred: EvalPrediction):
    logits, labels = eval_pred
    preds = logits.squeeze()

    # Regression metrics
    rmse = mean_squared_error(labels, preds, squared=False)
    mae = mean_absolute_error(labels, preds)
    r2 = r2_score(labels, preds)

    # Classification thresholds
    pred_classes = np.select(
        [preds < -0.33, preds > 0.33],
        [0, 2],  # negative, positive
        default=1  # neutral
    )
    true_classes = np.select(
        [labels < -0.33, labels > 0.33],
        [0, 2],
        default=1
    )

    precision = precision_score(true_classes, pred_classes, average='macro', zero_division=0)
    f1 = f1_score(true_classes, pred_classes, average='macro', zero_division=0)

    return {
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
        "precision": precision,
        "f1": f1
    }

In [46]:
training_args = TrainingArguments(
    output_dir="./output",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    warmup_steps=100,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    remove_unused_columns=True,
    load_best_model_at_end=True,
    logging_steps=50,
    fp16=True,
    dataloader_num_workers=4,
    save_total_limit=2,
    report_to="none"
)

class ProgressTrainer(Trainer):
    def train(self, *args, **kwargs):
        print("Starting training...")
        result = super().train(*args, **kwargs)
        print("Training completed.")
        return result

trainer = ProgressTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

#train model
trainer.train()

trainer.save_model("./output/best_model")
tokenizer.save_pretrained("./output/best_model_tok")

print(f"Model and tokenizer saved")



Starting training...


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Rmse,Mae,R2,Precision,F1
1,0.010300,0.008926,0.094475,0.069040,0.937768,0.881529,0.880491
2,0.006000,0.006716,0.081949,0.057478,0.953177,0.896013,0.901541
3,0.003900,0.006418,0.080114,0.058686,0.955250,0.895328,0.903733
4,0.002200,0.004618,0.067955,0.046200,0.967803,0.918219,0.921125
5,0.001400,0.004551,0.067462,0.046702,0.968268,0.915943,0.920729


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Training completed.
Model and tokenizer saved


In [4]:
!zip -r file.zip /kaggle/working/output/best_model

	zip warning: name not matched: /kaggle/working/output/best_model

zip error: Nothing to do! (try: zip -r file.zip . -i /kaggle/working/output/best_model)
